# Week 7 Assignment: Incremental Data Processing using Delta Lake (Superstore Dataset)

### Objectives:
1. Initialize a Spark session configured for Delta Lake.
2. Download the Superstore dataset using `kagglehub`.
3. Load, clean, and write the base Superstore dataset to a Delta Table.
4. Simulate new/incremental updates containing updates and inserts.
5. Apply a Delta Lake `MERGE` operation (SCD Type 1) to update modified records and append new records.
6. Validate merge results (key uniqueness and row count).

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, trim
from delta.tables import DeltaTable
import kagglehub
import os

# 1. Initialize Spark Session configured with Delta Lake
spark = SparkSession.builder \
    .appName("DeltaLakeSCDSuperstore") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

print("Spark Session initialized with Delta support.")

### Step 1: Download and Load Superstore Dataset
We download the latest `vivek468/superstore-dataset-final` using `kagglehub` and read the CSV into a PySpark DataFrame.

In [ ]:
# Download dataset from Kaggle
dataset_path = kagglehub.dataset_download("vivek468/superstore-dataset-final")
csv_file = os.path.join(dataset_path, "Sample - Superstore.csv")

# Read CSV file
raw_df = spark.read.csv(csv_file, header=True, inferSchema=True)
print(f"Dataset successfully loaded. Total rows: {raw_df.count()}")
raw_df.select("Row ID", "Order ID", "Customer ID", "Customer Name", "Sales", "Profit").show(5)

### Step 2: Perform Data Cleaning
We standardize column names to lowercase with underscores, handle nulls, and drop duplicates on the primary key (`row_id`).

In [ ]:
# Rename columns to standard snake_case
cleaned_df = raw_df \
    .withColumnRenamed("Row ID", "row_id") \
    .withColumnRenamed("Order ID", "order_id") \
    .withColumnRenamed("Customer ID", "customer_id") \
    .withColumnRenamed("Customer Name", "customer_name") \
    .withColumnRenamed("Segment", "segment") \
    .withColumnRenamed("City", "city") \
    .withColumnRenamed("Sales", "sales") \
    .withColumnRenamed("Profit", "profit")

# Select a subset of columns for clean demonstration
cleaned_df = cleaned_df.select("row_id", "order_id", "customer_id", "customer_name", "segment", "city", "sales", "profit")

# Handle duplicates on row_id and fill missing customer names
cleaned_df = cleaned_df \
    .dropDuplicates(["row_id"]) \
    .fillna({"customer_name": "Unknown Customer"})

print("Deduplicated and cleaned base schema:")
cleaned_df.show(5)

# Save as local Delta Table
delta_path = "/tmp/delta-lake/superstore_master_table"
cleaned_df.write.format("delta").mode("overwrite").save(delta_path)
print(f"Delta table written to: {delta_path}")

### Step 3: Create Incremental Data Updates
We create a secondary dataset simulating new transactions and updates to existing entries.

In [ ]:
# Simulate incremental batch:
# - Row ID 1: Updated Sales to 350.00 and Profit to 50.00
# - Row ID 2: Updated Sales to 800.00 and Profit to 120.00
# - Row ID 99999: New order insertion
incremental_data = [
    (1, "CA-2016-152156", "CG-12520", "Claire Gute", "Consumer", "Henderson", 350.00, 50.00),
    (2, "CA-2016-152156", "CG-12520", "Claire Gute", "Consumer", "Henderson", 800.00, 120.00),
    (99999, "CA-2026-999999", "SM-23180", "Suyash Maheshwari", "Corporate", "Jaipur", 120.00, 35.00)
]

columns = ["row_id", "order_id", "customer_id", "customer_name", "segment", "city", "sales", "profit"]
incremental_df = spark.createDataFrame(incremental_data, schema=columns)

print("Incremental Updates to Apply:")
incremental_df.show()

### Step 4: Apply Delta `MERGE` (SCD Type 1)
Using Delta Lake's native `MERGE` operation, we match on `row_id` to update matching records in place and insert new records.

In [ ]:
deltaTable = DeltaTable.forPath(spark, delta_path)

deltaTable.alias("target") \
  .merge(
    incremental_df.alias("source"),
    "target.row_id = source.row_id"
  ) \
  .whenMatchedUpdate(set = {
    "sales": "source.sales",
    "profit": "source.profit",
    "city": "source.city",
    "order_id": "source.order_id"
  }) \
  .whenNotMatchedInsert(values = {
    "row_id": "source.row_id",
    "order_id": "source.order_id",
    "customer_id": "source.customer_id",
    "customer_name": "source.customer_name",
    "segment": "source.segment",
    "city": "source.city",
    "sales": "source.sales",
    "profit": "source.profit"
  }) \
  .execute()

print("Delta MERGE complete.")

### Step 5: Validate Results
Verify the final merged dataset row counts and ensure `row_id` uniqueness.

In [ ]:
final_df = spark.read.format("delta").load(delta_path)

# Row count check (Original rows + 1 new row)
final_count = final_df.count()
print(f"Final Row Count in Merged Table: {final_count}")

# Verify updates on Row ID 1 and Row ID 2
print("Updated Target Records (Row ID 1 & 2):")
final_df.filter("row_id IN (1, 2)").show()

# Verify insert on Row ID 99999
print("Newly Inserted Record:")
final_df.filter("row_id = 99999").show()

# Uniqueness check
unique_count = final_df.select("row_id").distinct().count()
assert final_count == unique_count, "PrimaryKey Violation: Duplicates detected!"
print("Validation successful: Primary key integrity intact.")